# Raycaster — Project Walkthrough

A quick tour of the engine's internals, run headlessly (no tkinter
window needed here — the actual game window is launched separately with
`python -m raycaster`, see the README).

This notebook:
1. Loads a level and builds textures
2. Renders a single frame and displays it inline
3. Runs the trig accuracy/speed checks from `tools/benchmarks.py`
4. Runs the vault connectivity check from `tools/verify_connectivity.py`

Run this from the repo root so the `raycaster` package resolves without
needing `PYTHONPATH` set manually.

In [ ]:
import sys
from pathlib import Path

# Make sure the repo root is importable, same as PYTHONPATH=. in the README.
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt

from raycaster import level as level_mod
from raycaster import textures as textures_mod
from raycaster import render as render_mod
from raycaster import config
from raycaster.entities import GameState

## 1. Load the showcase level and build textures

Textures are procedurally generated and cached to `raycaster/assets_cache/`
on first run — see `textures.build_textures`. This step is what
`python -m raycaster` does on startup, minus the tkinter window.

In [ ]:
lvl = level_mod.load_level()
tex = textures_mod.build_textures('raycaster/assets_cache', lvl.wall_colors)
state = GameState.new_game(lvl, tex)

print(f'Level: {lvl.name}')
print(f'Grid: {lvl.map_rows} x {lvl.map_cols}')
print(f'Doors: {len(lvl.door_cells)}')
print(f'Sprites: {len(lvl.sprites)}')
print(f'Key pickups: {len(lvl.key_pickups)}')

## 2. Render a single frame

This calls the exact same `render()` / `draw_sprites()` functions used
in the live game loop, just without tkinter — the output is a plain
NumPy RGB buffer at the internal render resolution (`config.RENDER_W` x
`config.RENDER_H`), which we can display directly with matplotlib.

In [ ]:
buf = np.zeros((config.RENDER_H, config.RENDER_W, 3), dtype=np.uint8)
z = render_mod.render(state, buf)
render_mod.draw_sprites(state, buf, z)

plt.figure(figsize=(10, 6))
plt.imshow(buf)
plt.axis('off')
plt.title(f'Spawn view — {config.RENDER_W}x{config.RENDER_H} internal resolution')
plt.show()

## 3. Trig accuracy and speed

Same checks as `tools/benchmarks.py`, inline. `mc_sin`/`mc_cos` are the
scalar Taylor-series implementations used in a handful of places;
`vec_sin`/`vec_cos` are the vectorised forms used in the actual rotation
hot path. See the README's "Why a custom Taylor series" section for why
these exist instead of just calling `math.sin`/`np.sin`.

In [ ]:
import math
import time
from raycaster.math_utils import mc_sin, mc_cos, vec_sin, vec_cos

sweep = np.linspace(-math.pi, math.pi, 20000)

t0 = time.perf_counter()
scalar_sin_err = max(abs(mc_sin(x) - math.sin(x)) for x in sweep)
t_scalar_sin = time.perf_counter() - t0

t0 = time.perf_counter()
vec_sin_err = float(np.max(np.abs(vec_sin(sweep) - np.sin(sweep))))
t_vec_sin = time.perf_counter() - t0

print(f'mc_sin  max abs error: {scalar_sin_err:.3e}   time: {t_scalar_sin*1000:.2f} ms')
print(f'vec_sin max abs error: {vec_sin_err:.3e}   time: {t_vec_sin*1000:.2f} ms')
print()
print('Full accuracy/speed comparison including cos and stdlib timing:')
print('  PYTHONPATH=. python3 tools/benchmarks.py')

## 4. Vault connectivity (BFS)

Confirms the vault room is only reachable once the door is open — not
reachable some other way through the grid. Full version with the gated
cell list lives in `tools/verify_connectivity.py`.

In [ ]:
from collections import deque

def bfs_reachable_cells(lvl, door_open):
    grid = lvl.world_map
    rows, cols = lvl.map_rows, lvl.map_cols
    start = (int(lvl.spawn['px']), int(lvl.spawn['py']))

    def passable(r, c):
        if not (0 <= r < rows and 0 <= c < cols):
            return False
        v = grid[r, c]
        if v == 0:
            return True
        if v == config.DOOR_ID:
            return door_open
        return False

    visited = {start}
    q = deque([start])
    while q:
        r, c = q.popleft()
        for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
            nr, nc = r+dr, c+dc
            if (nr, nc) not in visited and passable(nr, nc):
                visited.add((nr, nc))
                q.append((nr, nc))
    return visited

closed = bfs_reachable_cells(lvl, door_open=False)
opened = bfs_reachable_cells(lvl, door_open=True)
gated = opened - closed

print(f'Reachable with doors CLOSED: {len(closed)} cells')
print(f'Reachable with doors OPEN:   {len(opened)} cells')
print(f'Cells gated behind a door:   {len(gated)} cells')

## Next steps

- Run the real game: `python -m raycaster`
- Run the full test suite: `PYTHONPATH=. python3 -m pytest tests/ -v`
- Full benchmark output: `PYTHONPATH=. python3 tools/benchmarks.py`

See `README.md` for architecture details, controls, and the level data
format.